In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from surprise import SVD, Dataset, Reader

In [2]:
# 1. LOAD & PREPARE DATASET
# ---------------------------------------------------------
# Waa in aad MovieLens 100K ama dataset u dhíganta oo 'movies.csv' iyo 'ratings.csv' ah haysataa.
movies = pd.read_csv('movies.csv')   # Columns: movieId, title, genres
ratings = pd.read_csv('ratings.csv') # Columns: userId, movieId, rating

print(f"Loaded {len(movies)} movies and {len(ratings)} ratings.")

Loaded 4803 movies and 100836 ratings.


In [3]:
# 2. CONTENT-BASED FILTERING (TF-IDF on Genres/Overview)
# ---------------------------------------------------------
movies['genres'] = movies['genres'].fillna('')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])

# Caluclate Cosine Similarity score between movies
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# Map movie titles to index
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def get_content_recommendations(title, top_n=10):
    """Fetches movies with similar features/genres."""
    if title not in indices:
        return []
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    return movies.iloc[movie_indices]

In [4]:
# 3. COLLABORATIVE FILTERING (SVD Matrix Factorization)
# ---------------------------------------------------------
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset = data.build_full_trainset()

# Train SVD Algorithm
svd = SVD()
svd.fit(trainset)

In [5]:
# 4. HYBRID RECOMMENDATION SYSTEM
# ---------------------------------------------------------
def hybrid_recommendation(user_id, title, top_n=10):
    """
    Combines Content-Based candidates with Collaborative predictions
    to produce personalized hybrid recommendations.
    """
    # Step A: Get content-based candidates (top 30)
    if title not in indices:
        print(f"Movie '{title}' not found in database.")
        return None

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:31]
    
    movie_candidates = []
    for i in sim_scores:
        m_id = movies.iloc[i[0]]['movieId']
        m_title = movies.iloc[i[0]]['title']
        
        # Step B: Predict user's rating for candidate using SVD
        predicted_rating = svd.predict(user_id, m_id).est
        movie_candidates.append((m_title, predicted_rating))

    # Step C: Sort candidates by predicted user rating
    movie_candidates = sorted(movie_candidates, key=lambda x: x[1], reverse=True)
    
    return pd.DataFrame(movie_candidates[:top_n], columns=['Recommended Movie', 'Predicted Rating'])

In [6]:
# 5. TEST THE HYBRID SYSTEM
# ---------------------------------------------------------
if __name__ == "__main__":
    sample_user = 1
    sample_movie = "Toy Story (1995)"
    
    print(f"\n--- Top Hybrid Recommendations for User {sample_user} (Based on '{sample_movie}') ---")
    recommendations = hybrid_recommendation(user_id=sample_user, title=sample_movie, top_n=5)
    print(recommendations)


--- Top Hybrid Recommendations for User 1 (Based on 'Toy Story (1995)') ---
Movie 'Toy Story (1995)' not found in database.
None
